In [ ]:

# ==========================================================
# VISIONAI STUDIO - COMPLETE APPLICATION
# AI IMAGE RECOGNITION USING BLIP + GRADIO
# ==========================================================

import gradio as gr
import torch
from PIL import Image, ImageOps
from transformers import BlipProcessor, BlipForConditionalGeneration

# ==========================================================
# 1. LOAD AI MODEL
# ==========================================================

device = "cuda" if torch.cuda.is_available() else "cpu"

if "processor" not in globals() or "model" not in globals():

    print("Loading VisionAI model...")

    processor = BlipProcessor.from_pretrained(
        "Salesforce/blip-image-captioning-base"
    )

    model = BlipForConditionalGeneration.from_pretrained(
        "Salesforce/blip-image-captioning-base"
    ).to(device)

    model.eval()

print("VisionAI is ready!")


# ==========================================================
# 2. IMAGE ANALYSIS FUNCTION
# ==========================================================

def analyze_image(image, category):

    if image is None:
        return None, "Please upload an image first.", "Waiting for image"

    try:

        image = ImageOps.exif_transpose(image).convert("RGB")

        inputs = processor(
            images=image,
            return_tensors="pt"
        ).to(device)

        with torch.inference_mode():

            output = model.generate(
                **inputs,
                max_new_tokens=55
            )

        description = processor.decode(
            output[0],
            skip_special_tokens=True
        )

        safe_description = (
            description.replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace('"', "&quot;")
            .replace("'", "&#39;")
        )

        safe_category = (
            str(category)
            .replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
        )

        result_html = f"""
        <div class="result-card">

            <div class="result-eyebrow">
                ✦ ANALYSIS COMPLETE
            </div>

            <h2>✨ Here's what I see</h2>

            <p class="description">
                {safe_description.capitalize()}
            </p>

            <div class="info-row">
                <span>Selected category</span>
                <strong>{safe_category}</strong>
            </div>

            <div class="info-row">
                <span>AI model</span>
                <strong>BLIP Vision</strong>
            </div>

            <div class="info-row">
                <span>Processing device</span>
                <strong>{device.upper()}</strong>
            </div>

            <p class="disclaimer">
                This is an AI-generated image description.
                The selected category is a preference and
                is not a verified classification.
            </p>

        </div>
        """

        return (
            image,
            result_html,
            "✓ Image analyzed successfully"
        )

    except Exception as e:

        return (
            image,
            f"<h3>Analysis failed</h3><p>{type(e).__name__}: {str(e)}</p>",
            "Something went wrong"
        )


# ==========================================================
# 3. PREMIUM DARK UI CSS
# ==========================================================

css = """

@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');

.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
    font-family: 'Inter', sans-serif !important;
    background: #090d1c !important;
    color: #ffffff !important;
}

body {
    background: #090d1c !important;
}

footer {
    display: none !important;
}

#hero {
    background:
        radial-gradient(circle at 85% 15%, #5141a8 0%, transparent 35%),
        linear-gradient(135deg, #171d3c, #101329 65%, #252052);

    padding: 48px;
    border-radius: 28px;
    color: white;
    margin-bottom: 28px;

    border: 1px solid #34345e;

    box-shadow: 0 20px 60px rgba(0,0,0,.3);
}

#brand {
    color: #b7aaff;
    font-size: 12px;
    font-weight: 800;
    letter-spacing: 3px;
    margin-bottom: 25px;
}

#hero h1 {
    font-size: clamp(38px, 6vw, 65px);
    line-height: 1.08;
    letter-spacing: -2px;
    color: white;
    margin-bottom: 18px;
}

#hero p {
    color: #c4c8e6;
    font-size: 16px;
    line-height: 1.9;
}

.panel {
    background: linear-gradient(145deg, #171d35, #11162b) !important;
    padding: 25px !important;
    border-radius: 23px !important;
    border: 1px solid #303653 !important;
    box-shadow: 0 12px 35px rgba(0,0,0,.2);
}

.panel h2,
.panel h3 {
    color: #ffffff !important;
}

.panel label {
    color: #d5d7ee !important;
}

.panel input,
.panel textarea {
    background: #10162b !important;
    border-color: #343b5a !important;
    color: white !important;
}

#analyze {
    border: none !important;
    border-radius: 14px !important;
    min-height: 56px;
    font-size: 16px !important;
    font-weight: 800 !important;

    background: linear-gradient(
        110deg,
        #6955ed,
        #a276ff
    ) !important;

    color: white !important;

    box-shadow: 0 8px 25px rgba(112, 85, 237, .3);

    transition: .25s;
}

#analyze:hover {
    transform: translateY(-2px);
    filter: brightness(1.15);
}

#result {
    background: #11172d !important;
    padding: 23px;
    border-radius: 18px;
    border: 1px solid #343b5a;
    min-height: 270px;
    color: white;
}

.result-card h2 {
    font-size: 25px;
    color: #ffffff;
    margin: 15px 0 20px;
}

.result-eyebrow {
    color: #a99aff;
    font-weight: 800;
    font-size: 11px;
    letter-spacing: 2px;
}

.description {
    background: linear-gradient(135deg, #252447, #1c203c);
    padding: 20px;
    border: 1px solid #39365f;
    border-radius: 14px;
    line-height: 1.9;
    font-size: 16px;
    color: #e5e3ff;
}

.info-row {
    display: flex;
    justify-content: space-between;
    gap: 10px;
    border-bottom: 1px solid #303650;
    padding: 15px 0;
    font-size: 13px;
}

.info-row span {
    color: #999fbd;
}

.info-row strong {
    color: #c4b5ff;
    text-align: right;
}

.disclaimer {
    font-size: 12px;
    color: #8f96b5;
    line-height: 1.8;
    margin-top: 20px;
}

#status textarea {
    text-align: center;
    font-weight: 700;
}

@media(max-width: 700px) {

    #hero {
        padding: 25px;
        border-radius: 20px;
    }

    .panel {
        padding: 15px !important;
    }

}

"""


# ==========================================================
# 4. BUILD VISIONAI WEBSITE
# ==========================================================

with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="violet",
        secondary_hue="indigo"
    ),
    css=css,
    title="VisionAI | AI Image Recognition"
) as demo:

    # HERO SECTION

    gr.HTML("""
    <div id="hero">

        <div id="brand">
            ✦ VISIONAI STUDIO
        </div>

        <h1>
            Discover the<br>
            world through AI.
        </h1>

        <p>
            Upload any image and let artificial intelligence
            understand what's inside it. Fast, simple and intelligent.
        </p>

        <p>
            🐾 Animals &nbsp;&nbsp;
            🍎 Fruits &nbsp;&nbsp;
            💻 Electronics &nbsp;&nbsp;
            🔤 Alphabet &nbsp;&nbsp;
            🌿 Nature
        </p>

    </div>
    """)

    # MAIN CONTENT

    with gr.Row(equal_height=True):

        # LEFT SIDE - IMAGE UPLOAD

        with gr.Column(
            scale=5,
            elem_classes=["panel"]
        ):

            gr.Markdown("## 📤 Upload Image")

            image_input = gr.Image(
                type="pil",
                label="Drop your image here",
                height=300
            )

            category_input = gr.Dropdown(
                choices=[
                    "All Categories",
                    "Animals",
                    "Fruits",
                    "Electronics",
                    "Alphabet",
                    "Vehicles",
                    "Flowers and Nature",
                    "Other Objects"
                ],
                value="All Categories",
                label="Select image category"
            )

            analyze_button = gr.Button(
                "✦  Analyze Image with AI",
                variant="primary",
                elem_id="analyze"
            )

            clear_button = gr.ClearButton(
                components=[image_input],
                value="↻ Clear Image"
            )

            gr.Markdown("""
            ### 💡 Tips for better results

            - Use clear, high-quality images.
            - Keep the main object visible.
            - Avoid excessive background clutter.
            - Use readable images for alphabet recognition.
            """)

        # RIGHT SIDE - AI RESULTS

        with gr.Column(
            scale=6,
            elem_classes=["panel"]
        ):

            gr.Markdown("## ✨ AI Recognition")

            image_preview = gr.Image(
                label="Image Preview",
                height=220,
                interactive=False
            )

            result_output = gr.HTML(
                value="""
                <div class="result-card">

                    <div class="result-eyebrow">
                        YOUR AI WORKSPACE
                    </div>

                    <h2>Ready to explore? 🚀</h2>

                    <p class="description">
                        Your results will appear here.
                        Upload an image and click the Analyze
                        button to get an AI-generated description.
                    </p>

                    <div class="info-row">
                        <span>AI Engine</span>
                        <strong>BLIP Vision</strong>
                    </div>

                    <div class="info-row">
                        <span>Status</span>
                        <strong>Waiting for image</strong>
                    </div>

                </div>
                """,
                elem_id="result"
            )

            status_output = gr.Textbox(
                value="● System ready",
                label="System Status",
                interactive=False,
                elem_id="status"
            )

    # FOOTER

    gr.HTML("""
    <div style="
        text-align:center;
        padding:30px 10px;
        color:#858ba9;
        font-size:12px;
        line-height:2;">

        <b style="color:#a99aff;letter-spacing:2px">
            ✦ VISIONAI STUDIO
        </b>

        <br>

        Powered by Python, Gradio and BLIP AI

        <br>

        Intelligent image descriptions · Built with AI

    </div>
    """)

    # BUTTON FUNCTIONALITY

    analyze_button.click(
        fn=analyze_image,
        inputs=[
            image_input,
            category_input
        ],
        outputs=[
            image_preview,
            result_output,
            status_output
        ]
    )


# ==========================================================
# 5. LAUNCH APPLICATION
# ==========================================================

demo.queue()

demo.launch(
    share=True,
    show_error=True
)


Loading VisionAI model...


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

VisionAI is ready!


/tmp/ipykernel_2991/2539724841.py:323: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://03abaf463b070b249a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
